# 🧬 Protein Design Workshop — Team Activity
### Three teams, three proteins, one workflow.

Everyone runs the same steps. Each team works on a different protein. We'll compare notes at the end.

| Team | Protein | Lesson |
|---|---|---|
| **1** — Easy / industrial | **PETase** (*Ideonella sakaiensis*) | What protein design looks like when the data is good and the problem is tractable. |
| **2** — Plant / hard | **ZAR1** (*Arabidopsis thaliana*) | What happens when the protein is large, dynamic, and from an under-represented training-data lane. |
| **3** — Schmidt relevant / real-world | **CarRP** (*Mucor circinelloides*) | What design feels like on a real industrial target — no experimental structure, non-model fungus, bifunctional enzyme. |

**Pick your team by editing the `TEAM` variable in the cell below.** Michael will walk everyone through the same steps; you'll get different numbers on your screen, and we'll debrief what each team saw.

**What you'll do:**
1. Choose your team's protein and load its structure
2. Run ProteinMPNN to design new sequences that fold to the same shape
3. Validate with AlphaFold — does the designed sequence actually fold as intended?
4. Visualize the design vs. the AF2 prediction
5. Compare across teams — what did each protein teach us?

---

## ✅ Step 1: Verify environment
Run this once to make sure everything's installed on your AWS instance.

In [ ]:
# Environment check
import os, sys, torch

ok = True

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU detected."); ok = False

try:
    from colabdesign.mpnn import mk_mpnn_model
    from colabdesign.af   import mk_af_model
    print("✅ ColabDesign (ProteinMPNN + AF2)")
except ImportError as e:
    print(f"❌ ColabDesign: {e}"); ok = False

try:
    import py3Dmol
    print("✅ py3Dmol")
except ImportError:
    print("❌ py3Dmol"); ok = False

af_params = os.path.expanduser("~/params/params/params_model_1.npz")
print(f"{'✅' if os.path.isfile(af_params) else '❌'} AlphaFold2 params")
if not os.path.isfile(af_params): ok = False

if ok:
    print("\n🎉 Ready to go.")
else:
    print("\n⚠️  Fix the errors before continuing. See setup.sh.")


## 🎯 Step 2: Pick your team's protein

Edit `TEAM` to `"PETase"`, `"ZAR1"`, or `"CarRP"` and run the cell.

In [ ]:
# =============================================================================
# THE THREE TEAM TARGETS
# =============================================================================
# Pick ONE by setting TEAM = "PETase", "ZAR1", or "CarRP"
# Each team works in parallel on the same steps with a different protein.
# =============================================================================

TEAM = "PETase"  # <-- EDIT THIS: "PETase", "ZAR1", or "CarRP"

TARGETS = {
    "PETase": {
        "label":      "PETase (easy / industrial)",
        "source":     "pdb",
        "pdb_id":     "6EQE",
        "chain":      "A",
        "range":      None,                     # use full chain
        "organism":   "Ideonella sakaiensis (bacterium)",
        "function":   "Plastic (PET) depolymerase",
        "hotspots":   "A159,A161,A185",         # catalytic-face residues for binder design
        "why":        (
            "The 'easy' case. Well-characterized, single-domain α/β hydrolase (~290 aa). "
            "Plenty of training data from related cutinases and esterases. This is the "
            "protein-design equivalent of a clean test set — AI models should look good here."
        ),
        "preamble_tie": (
            "Industrial biomanufacturing, slide 10. PETase is the poster child for "
            "designed enzymes in the circular bioeconomy — FAST-PETase was engineered "
            "via ML in 2022. Your team will feel what protein design looks like when "
            "the data is good and the problem is tractable."
        ),
    },
    "ZAR1": {
        "label":      "ZAR1 (plant immune receptor / hard case)",
        "source":     "pdb",
        "pdb_id":     "6J5T",
        "chain":      "A",
        "range":      (1, 200),                 # truncate to CC + NB-ARC (~200 aa)
        "organism":   "Arabidopsis thaliana (plant)",
        "function":   "NLR immune receptor — detects bacterial effectors, forms pentameric 'resistosome'",
        "hotspots":   "A14,A17,A24",            # CC-domain surface — proposed effector face
        "why":        (
            "The 'hard' case. Plant immune receptors are large, multidomain, and form "
            "dynamic oligomeric complexes (the ZAR1 resistosome is a pentamer that "
            "punctures cell membranes). Plant proteins are significantly under-represented "
            "in training sets vs. bacterial/human proteins. We truncate to the CC + NB-ARC "
            "domain (~200 aa) to keep the workshop tractable — but the full biology is "
            "exactly the kind of thing current models struggle with."
        ),
        "preamble_tie": (
            "Limitations slide (generalization + dynamics). ZAR1 exercises two failure "
            "modes at once — it's a plant protein (training-data asymmetry) AND it's a "
            "conformationally dynamic oligomer (static-structure bias). Your team will feel "
            "the edges of current capability."
        ),
    },
    "CarRP": {
        "label":      "CarRP (Schmidt Sciences relevant / industrial fungal)",
        "source":     "afdb",                   # AlphaFold Database — no experimental structure
        "uniprot":    "Q9UUQ6",
        "chain":      "A",
        "range":      (1, 330),                 # R domain = lycopene cyclase (functional alone)
        "organism":   "Mucor circinelloides (fungus)",
        "function":   "Bifunctional lycopene cyclase + phytoene synthase — carotenoid biosynthesis",
        "hotspots":   "A45,A80,A150",           # cyclase-domain face (approximate)
        "why":        (
            "The 'real world' case. CarRP is a fungal bifunctional enzyme with no "
            "experimental structure — we use an AlphaFold Database prediction. "
            "This is what most industrially-relevant targets actually look like when "
            "you try to design on them. Fungi are under-represented in training data. "
            "The bifunctional architecture (lycopene cyclase + phytoene synthase fused "
            "into one polypeptide) is rare and challenging."
        ),
        "preamble_tie": (
            "Industrial biomanufacturing (slide 10) + the PLAID-Bio thesis (data slide, "
            "slide 14). Carotenoids are a ~$1.8B global market — CarRP sits at the heart "
            "of microbial lycopene/β-carotene production. Your team will feel what "
            "protein design is like on a real non-model-organism industrial target."
        ),
    },
}

if TEAM not in TARGETS:
    raise ValueError(f"Unknown TEAM '{TEAM}'. Options: {list(TARGETS)}")

target = TARGETS[TEAM]
print("=" * 70)
print(f"🧬 Team {TEAM}: {target['label']}")
print("=" * 70)
print(f"Organism:   {target['organism']}")
print(f"Function:   {target['function']}")
print()
print(f"Why this protein:")
print(f"  {target['why']}")
print()
print(f"Preamble tie-in:")
print(f"  {target['preamble_tie']}")
print("=" * 70)

## 📥 Step 3: Load the target structure

This downloads the structure for your team's protein. Notice the source:
- **PETase** and **ZAR1** come from the RCSB PDB (experimental structures).
- **CarRP** comes from the **AlphaFold Database** (predicted structure). This is how most industrial targets arrive in practice — no one has crystallized them.

In [ ]:
# Load the target protein structure
import os, subprocess

os.makedirs("inputs", exist_ok=True)
input_pdb = f"inputs/{TEAM}.pdb"

if target["source"] == "pdb":
    # Download from RCSB
    url = f"https://files.rcsb.org/download/{target['pdb_id']}.pdb"
    subprocess.run(f"wget -q {url} -O {input_pdb}", shell=True, check=True)
    print(f"✅ Downloaded PDB {target['pdb_id']} from RCSB")
elif target["source"] == "afdb":
    # Download AlphaFold prediction
    url = f"https://alphafold.ebi.ac.uk/files/AF-{target['uniprot']}-F1-model_v4.pdb"
    subprocess.run(f"wget -q {url} -O {input_pdb}", shell=True, check=True)
    print(f"✅ Downloaded AlphaFold prediction for UniProt {target['uniprot']} (CarRP)")
    print("   Note: This is a PREDICTED structure, not an experimental one.")
    print("         Many industrial targets only have AlphaFold predictions available.")

# Truncate if a range is specified (keep only the relevant domain)
if target["range"] is not None:
    lo, hi = target["range"]
    trunc = f"inputs/{TEAM}_trunc.pdb"
    with open(input_pdb) as fin, open(trunc, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM", "HETATM")):
                try:
                    resnum = int(line[22:26])
                    if resnum < lo or resnum > hi:
                        continue
                except ValueError:
                    pass
                if line[21] != target["chain"] and line[21] != " ":
                    continue
            fout.write(line)
    input_pdb = trunc
    print(f"✂️  Truncated to chain {target['chain']} residues {lo}-{hi}")

# Report size
n_res = len({line[22:26] for line in open(input_pdb)
             if line.startswith("ATOM") and line[21] == target["chain"]})
print(f"📏 Working structure: {n_res} residues")

## ✍️ Step 4: Design new sequences with ProteinMPNN

ProteinMPNN looks at the backbone of your protein and writes new sequences that should fold to that same backbone. This is **inverse folding** — the 2022 foundation of modern protein design.

We'll generate 4 candidate sequences per team and compare their ProteinMPNN scores (lower = more confident the design is sensible).

In [ ]:
from colabdesign.mpnn import mk_mpnn_model

mpnn = mk_mpnn_model()
mpnn.prep_inputs(pdb_filename=input_pdb, rm_aa="C")   # exclude Cys to avoid disulfides

print(f"Designing sequences for {TEAM}...\n")
out = mpnn.sample(num=4, temperature=0.1)

designs = []
for i, seq in enumerate(out["seq"]):
    clean = seq.replace("/", "")
    score = float(out["score"][i])
    designs.append({"idx": i, "sequence": clean, "score": score})
    print(f"  Design {i+1}: score={score:.3f}  len={len(clean)}")
    print(f"    {clean[:70]}{'...' if len(clean) > 70 else ''}")

print(f"\n✅ Generated {len(designs)} ProteinMPNN designs for team {TEAM}.")


## 🔬 Step 5: Validate with AlphaFold

Does each designed sequence actually fold into the shape we wanted?

AlphaFold predicts structure from sequence alone. We feed it the **designed sequence** and check:
- **pLDDT** — AlphaFold's confidence in its prediction. >0.7 is good.
- **RMSD** — how far the AlphaFold prediction is from the original backbone. <2.5 Å is self-consistent.

Designs that pass both checks are the ones you'd make in the lab.

In [ ]:
from colabdesign.af import mk_af_model

af = mk_af_model(protocol="fixbb", use_templates=False, num_recycles=3)

validated = []
for d in designs:
    af.prep_inputs(pdb_filename=input_pdb, chain=target["chain"])
    af.set_seq(seq=d["sequence"])
    af.predict(num_recycles=3, verbose=False)

    plddt = float(af.aux["log"]["plddt"])
    rmsd  = float(af.aux["log"].get("rmsd", -1))
    ptm   = float(af.aux["log"].get("ptm", -1))

    d["plddt"] = plddt
    d["rmsd"]  = rmsd
    d["ptm"]   = ptm

    af_pdb = f"outputs_{TEAM}_{d['idx']}_af.pdb"
    os.makedirs("outputs", exist_ok=True)
    af_path = f"outputs/{af_pdb}"
    af.save_pdb(af_path)
    d["af_pdb"] = af_path

    consistent = plddt > 0.7 and (rmsd < 2.5 if rmsd >= 0 else True)
    verdict = "✅ self-consistent" if consistent else "⚠️  divergent"
    print(f"  Design {d['idx']+1}: pLDDT={plddt:.2f}  RMSD={rmsd:.2f}Å  pTM={ptm:.2f}   {verdict}")
    validated.append(d)

print(f"\n✅ Team {TEAM} results in. Pick your favorite design in the next cell.")


## 👁️ Step 6: Visualize

Gray = your original PETase / ZAR1 / CarRP backbone.
Colored = the AlphaFold prediction of the designed sequence (blue = confident, red = uncertain).

They should superimpose tightly if the design is self-consistent.

In [ ]:
import py3Dmol

WHICH = 1   # which of the 4 designs to show (1-indexed)

d = validated[min(WHICH - 1, len(validated) - 1)]

print(f"Team {TEAM} — Design {d['idx']+1}")
print(f"  pLDDT: {d['plddt']:.2f}  RMSD: {d['rmsd']:.2f}Å  pTM: {d['ptm']:.2f}")
print(f"  Sequence: {d['sequence']}")

with open(input_pdb)    as f: orig = f.read()
with open(d["af_pdb"])  as f: pred = f.read()

view = py3Dmol.view(width=900, height=500)
view.addModel(orig, "pdb")
view.addModel(pred, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray"}})
view.setStyle({"model": 1}, {"cartoon": {"colorscheme": {"prop": "b",
                                                         "gradient": "roygb",
                                                         "min": 50, "max": 90}}})
view.zoomTo()
view.show()


## 🗣️ Step 7: Debrief — compare across teams

Each team saw different numbers. Share with the room:

1. **What was your team's best pLDDT and RMSD?**
2. **Did you notice anything weird?** — low confidence in certain regions, divergent predictions, strange sequence preferences (lots of hydrophobic residues, lots of prolines, etc.)
3. **How does this match what you'd expect from the preamble?**

### What you'll probably see (don't read until you've run your team's results!)

<details>
<summary>Click to reveal expected findings</summary>

**Team PETase:** Clean results. pLDDT likely > 0.8 on all designs, RMSD < 1.5Å, self-consistent. This is what AI protein design looks like on a well-trained target. PETase is a small, single-domain α/β hydrolase; there are thousands of relatives in the PDB. The model has seen this architecture extensively.

**Team ZAR1:** More variable. Some designs may be self-consistent, others divergent. pLDDT likely drops in the CC-domain helices and at the truncation boundary. You're feeling the fact that plant proteins (and especially plant NLRs) are under-represented in training data, and that the CC domain is conformationally flexible.

**Team CarRP:** The most variable of the three. You started from an AlphaFold-predicted structure, not an experimental one. The truncation boundary (end of the R/cyclase domain) may have awkward geometry that ProteinMPNN struggles with. Fungal enzymes are less common in the PDB than bacterial ones. Welcome to the reality of design on real industrial targets.

**The meta-lesson:** The preamble argued that protein design's impact varies by lane — biomedical >> basic biology > industrial. What you just experienced is *why*: the underlying models perform best where training data is richest, and that correlates strongly with biomedical relevance. Closing that gap for industrial and environmental proteins is the PLAID-Bio thesis.

</details>

---

### Next: the bonus activity

Open `RFdiffusion_Industrial_Demo.ipynb` to design **novel binders** to your team's target. This uses RFdiffusion — the generative diffusion model that can create entirely new protein shapes from noise. You'll design a small protein that binds to your PETase / ZAR1 / CarRP.

The same three-team split applies. Same lessons, one level deeper.
